# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/martindiarua/ML_01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My rule and its reason codes

The baseline rule ranks content pages by a simple refresh-priority score using three signals supported by the ML-06 audit:

- **CTR:** stronger directional signal in this audit, because lower CTR was associated with worse average search position.
- **Staleness:** supporting signal, because older pages generally showed lower engagement, although the relationship was mixed.
- **Search volume:** supporting opportunity signal, because higher-volume pages showed slightly higher average impressions.

Each page receives points for showing one or more of these conditions. The total score determines its position in the ranked queue.

#### Scoring rule

- +2 points if CTR is in the lowest quartile.
- +1 point if the page has not been updated for more than 90 days.
- +1 point if search volume is above 20.

Pages are ranked from highest to lowest score.

#### Reason codes

- `LOW_CTR` — the page has relatively low CTR.
- `STALE` — the page has not been updated recently.
- `HIGH_VOLUME` — the page has relatively high search volume.

#### Action labels

- `REFRESH_REVIEW` — high-priority page that should be reviewed for a possible refresh.
- `MONITOR` — moderate-priority page that may need monitoring.
- `NO_ACTION` — lower-priority page based on these signals.

This is a baseline decision-support rule, not evidence that refreshing a page will improve its performance.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Ranked queue

I will apply the baseline rule to each content page and calculate a refresh-priority score.

The score uses only information available at the decision moment: CTR, days since the last update, and search volume. No future-window information or label-derived features are used.

The output will contain the content ID, score, reason code, action label, and the signals used to calculate the score.

The queue will be sorted from highest to lowest score and written to:

`work/outputs/baseline_action_score.csv`

In [1]:
!git clone https://github.com/martindiarua/ML_01.git
%cd ML_01/data/raw

Cloning into 'ML_01'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (157/157), done.
remote: Total 202 (delta 95), reused 99 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 2.27 MiB | 12.68 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/ML_01/data/raw


In [2]:
import os
import pandas as pd

# Load the dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Keep only the fields needed for the baseline
baseline = df[
    [
        "content_id",
        "ctr",
        "days_since_last_update",
        "search_volume"
    ]
].copy()

# Missing values cannot contribute evidence to the rule
baseline["ctr"] = baseline["ctr"].fillna(0)
baseline["search_volume"] = baseline["search_volume"].fillna(0)

# Start the score at zero
baseline["score"] = 0

# Apply the three rule components
baseline.loc[baseline["ctr"] <= 0.07, "score"] += 2
baseline.loc[baseline["days_since_last_update"] > 90, "score"] += 1
baseline.loc[baseline["search_volume"] > 20, "score"] += 1

# Reason codes
def get_reason(row):
    reasons = []

    if row["ctr"] <= 0.07:
        reasons.append("LOW_CTR")

    if row["days_since_last_update"] > 90:
        reasons.append("STALE")

    if row["search_volume"] > 20:
        reasons.append("HIGH_VOLUME")

    return "+".join(reasons) if reasons else "NONE"

baseline["reason_code"] = baseline.apply(get_reason, axis=1)

# Create action labels
baseline["action"] = baseline["score"].map(
    lambda x:
        "REFRESH_REVIEW" if x >= 3
        else "MONITOR" if x >= 1
        else "NO_ACTION"
)

# Rank the queue
baseline = baseline.sort_values(
    by=["score", "content_id"],
    ascending=[False, True]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# Output directory
os.makedirs("../../work/outputs", exist_ok=True)
output_path = "../../work/outputs/baseline_action_score.csv"
baseline.to_csv(output_path, index=False)

print("Ranked queue created:", output_path)
print("Rows:", len(baseline))

print("\nAction distribution:")
print(baseline["action"].value_counts())

print("\nTop 10:")
display(baseline.head(10))

Ranked queue created: ../../work/outputs/baseline_action_score.csv
Rows: 30000

Action distribution:
action
MONITOR           15802
NO_ACTION          7352
REFRESH_REVIEW     6846
Name: count, dtype: int64

Top 10:


,content_id,ctr,days_since_last_update,search_volume,score,reason_code,action,rank
0,content_006f466746a7,0.00,104,260.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,1
1,content_008fa457324e,0.00,104,50.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,2
2,content_00cd5e91c97a,0.00,104,210.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,3
3,content_0159bdc5c49f,0.00,104,320.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,4
4,content_01e609bf549b,0.00,104,40.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,5
5,content_02866c352877,0.00,104,70.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,6
6,content_02ae9b37d7b7,0.02,104,70.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,7
7,content_02b0d6e30129,0.00,313,110.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,8
8,content_02bcf3eec147,0.00,104,50.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,9
9,content_02c4842fefe7,0.00,98,390.0,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 pages are reviewed manually rather than treated as automatically correct.

For each page, I record:

- the recommended action,
- the reason code,
- a confidence note,
- and what would make the recommendation wrong.

The purpose is to check whether the simple rule produces sensible priorities rather than assuming that a high score automatically means a page should be refreshed.

Because the baseline uses only three directional signals, a high-ranked page could still be a weak recommendation. For example, low CTR may have another explanation unrelated to content freshness, while staleness does not prove that a refresh will improve performance.

In [3]:
top20 = baseline.head(20).copy()

display(
    top20[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action",
            "ctr",
            "days_since_last_update",
            "search_volume"
        ]
    ]
)

,rank,content_id,score,reason_code,action,ctr,days_since_last_update,search_volume
0,1,content_006f466746a7,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,104,260.0
1,2,content_008fa457324e,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,104,50.0
2,3,content_00cd5e91c97a,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,104,210.0
3,4,content_0159bdc5c49f,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,104,320.0
4,5,content_01e609bf549b,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,104,40.0
5,6,content_02866c352877,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,104,70.0
6,7,content_02ae9b37d7b7,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.02,104,70.0
7,8,content_02b0d6e30129,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,313,110.0
8,9,content_02bcf3eec147,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,104,50.0
9,10,content_02c4842fefe7,4,LOW_CTR+STALE+HIGH_VOLUME,REFRESH_REVIEW,0.00,98,390.0


### Review finding


The baseline successfully identifies pages where all three chosen signals agree, but it has limited ability to prioritize among pages with the same score. In this top-20 group, every page scores 4, so the current rule treats a page with search volume of 1,600 similarly to one with search volume of 40. This is a limitation of the simple baseline and provides a useful target for the later ML model to improve.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

The weakest picks are pages that receive a high score even though the available evidence may not clearly justify a refresh.

For example, a page may rank highly because it has low CTR and high search volume, but the low CTR may have another explanation that is not captured by this baseline. Similarly, staleness does not prove that a refresh will improve performance.

I will inspect the highest-ranked pages for these cases and note any recommendations that appear questionable.

### Leakage check

The baseline uses only three observed page-level signals:

- `ctr`
- `days_since_last_update`
- `search_volume`

I did not use the `refresh_priority` proxy label or the `leak_feature` created during the ML-05 leakage demonstration. I also did not use future-window performance fields.

The baseline therefore does not use the outcome label or a deliberately label-derived feature to calculate its score. This keeps the rule as a decision-support baseline rather than a leaked prediction.

In [4]:
# Features actually used by the baseline
baseline_features = [
    "ctr",
    "days_since_last_update",
    "search_volume"
]

print("Features used by baseline:")
print(baseline_features)

# Check that label-derived features were not used
forbidden_features = [
    "refresh_priority",
    "leak_feature"
]

print("\nLabel-derived leakage check:")

for column in forbidden_features:
    print(f"{column}: {column in baseline.columns}")

# Check that no future-window fields are part of the baseline
future_window_fields = [
    "impressions_future",
    "clicks_future",
    "sessions_future",
    "refresh_priority",
    "leak_feature"
]

print("\nleakage check:")

for column in future_window_fields:
    print(f"{column}: {column in baseline.columns}")

print("\nBaseline output columns:")
print(baseline.columns.tolist())

Features used by baseline:
['ctr', 'days_since_last_update', 'search_volume']

Label-derived leakage check:
refresh_priority: False
leak_feature: False

leakage check:
impressions_future: False
clicks_future: False
sessions_future: False
refresh_priority: False
leak_feature: False

Baseline output columns:
['content_id', 'ctr', 'days_since_last_update', 'search_volume', 'score', 'reason_code', 'action', 'rank']


In [5]:
from google.colab import userdata

# Navigate to the root of the repository
%cd /content/ML_01

# Retrieve the token securely from Colab Secrets
token = userdata.get('GITHUB_TOKEN')

# Configure your Git environment
!git config --global user.email "diaruamartin@gmail.com"
!git config --global user.name "Martin Diarua"

# Force add the CSV file to bypass any .gitignore rules
!git add -f work/outputs/baseline_action_score.csv

# Commit the file
!git commit -m "Upload baseline action score CSV to work/outputs"

# Push to GitHub
repo_url = f"https://{token}@github.com/martindiarua/ML_01.git"
!git push {repo_url} main

/content/ML_01
[main f88148f] Upload baseline action score CSV to work/outputs
 1 file changed, 30001 insertions(+)
 create mode 100644 work/outputs/baseline_action_score.csv
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 380.95 KiB | 126.98 MiB/s, done.
Total 5 (delta 2), reused 2 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/martindiarua/ML_01.git
   f516236..f88148f  main -> main


## Self-check

Before you submit, confirm each line honestly:

- [✔️] Every section above is filled — markdown thinking AND the code that backs it
- [✔️] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✔️] No client names, URLs, or private queries anywhere
- [✔️] My claims use careful words: observed, measured, directional, decision-support
- [✔️] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.